In [ ]:
# MWE for running MonteCarlo for different Hamiltonian models

from scipy.linalg import eigh
import numpy as np
from pyscf import gto, scf
from openfermion import hamiltonians
from qarp.operators.compat import from_openfermion

from qarp.operators import JordanWigner
from qarp.operators.integrals import restricted_integrals_to_fermion_operator
from qarp.operators.ucc import ucc_singles_and_doubles
from qarp.blocks import MappedONVStateBlock, TrotterAnsatzBlock, CompositeBlock
from qarp.algorithms import VQE
from qarp.algorithms import TermwiseHadamardTest, StateVector
from qarp.optimizers import ScipyOptimizer
from qarp.engines import QarpEngine

from qarp.algorithms import MonteCarlo, WalkerState
from qarp.endianness import bits_to_label, label_to_bits
import qarpx as qx
from qarp.operators import FullyCommuting, NoGrouping
from qarp.operators.pyscf import fermion_operator_from_mf, active_space_from_mf, onv_from_mf


QC-QMC:

QC-QMC (Quantum Computing Quantum Monte Carlo) is a semiclassical algorithm that allows to find the ground state of a quantum system by exploiting the Imaginary Time Evolution (ITE), and by using quantum computing for access information about the dynamics. 

The main differences with the standard Quantum Monte Carlo, can be found in two main ingredients:


1. The initial ansatz is obtained from a Unitary Transformation of the Hamiltonian, which in principle reduce the search space, by describing the system in a sub-space of the Hilbert space containing the real ground state. This process can be realized in various way, for instance with VQE, as we are doing in the following.

2. The QITE is realized by spawning walkers (quantum states) from a initial distribution. The walkers spawned and/or killed are determined by the transition elements of the Hamiltonian in the new basis (see point 1.). These transitions can be evaluated via quantum circuits (Hadamard test).

Here we show a semi-classical version of QCQMC where only point 1. is implemented. 


## Hamiltonian and initial state preparation

#### Fermi-Hubbard

<div style="background-color: #fcc651e8; border-left: 4px solid #ff9900ff; padding: 6px; margin: 10px 0; color: black;">
<strong>⚠️ Note:</strong> With this example, we use MonteCarlo to improve over the VQE convergence
</div>

In [ ]:
HH=hamiltonians.fermi_hubbard(
    2,
    2,
    -0.9,
    0.05,
    chemical_potential=0.1,
    magnetic_field=0.0,
    periodic=True,
    spinless=True,
    particle_hole_symmetry=False
)
HH = from_openfermion(HH)  # native boundary
qop = JordanWigner().encode_operator(HH)
qop_matrix = qop.sparse_matrix().toarray()

ham_mat = HH.sparse_matrix().toarray()
eg, en = np.linalg.eigh(ham_mat)

gs_opt = en[:, np.argmin(eg)]
gs_e = eg[np.argmin(eg)]
print("Exact GS energy: ", gs_e)

a = np.array([1]*3+ [0]*1)
np.random.shuffle(a)
print("Initial ONV: ", a)

onv = [int(x) for x in a]
fermionic_excitations, symbols = ucc_singles_and_doubles(onv)
qubit_excitations = JordanWigner().encode_operator(fermionic_excitations)

blocks = [MappedONVStateBlock(onv, JordanWigner()),
        TrotterAnsatzBlock(
            len(onv),
            qubit_excitations,
            symbols,
            steps=1,
            time=1,
            order=2,
            grouping=NoGrouping(),
            imaginary=True
        )
    ]
ket = CompositeBlock(blocks).build()
ket.plot(decompose_boxes=True, scrollable=True)

# VQE
initial_parameters = np.random.random(len(ket.symbols))
options = {"maxiter": 500}
optimizer = ScipyOptimizer(method="COBYLA", options=options)
vqe = VQE(
        operator=qop,
        ket=ket,
        primitive=StateVector(),
        engine=QarpEngine(),
        initial_parameters=initial_parameters,
        optimizer=optimizer,
        verbose=False,
).build()

e_vqe, x_vqe = vqe.run()

ham_mat = HH.sparse_matrix().toarray()

eigenvalues, eigenvectors = eigh(ham_mat)
h2_gs = eigenvectors[:, np.argmin(eigenvalues)]
h2_gse = eigenvalues[np.argmin(eigenvalues)]

# Overlap of VQE with exact GS (sparse_matrix and the qarpx statevector share the LSB basis)
params_vqe = vqe.optimal_parameters  # result.x mapped onto the sorted symbol tuple (§17)
h2_wfn_vqe = ket.set_symbols(params_vqe)
h2_wfn_vqe_SV = np.array(qx.QarpSimulator().statevector(h2_wfn_vqe.flatten(), h2_wfn_vqe.n_qubits))

print("overlap of VQE with exact GS: ", h2_wfn_vqe_SV @ h2_gs)

#### $H_2$ molecule

<div style="background-color: #fcc651e8; border-left: 4px solid #ff9900ff; padding: 6px; margin: 10px 0; color: black;">
<strong>⚠️ Note:</strong> With this example, we use MonteCarlo to follow up to a non-converged VQE result
</div>

In [ ]:
# Build H2 molecule and get UCCSD ansatz

bl = 0.635
geometry = f"H 0 0 0; H 0 0 {bl}"
mol = gto.M(atom=geometry, basis="sto3g", verbose=-1, symmetry=True)
mol.build()
mf = scf.RHF(mol)
mf.kernel()

onv = onv_from_mf(mf)
fermionic_uccsd, symbols = ucc_singles_and_doubles(
        onv, spin_conserving=True, generalised=False
)
qubit_uccsd = JordanWigner().encode_operator(fermionic_uccsd)

fermion_operator = fermion_operator_from_mf(mf)
h2_ham = JordanWigner().encode_operator(fermion_operator)#JordanWigner encoding for the Hamiltonian
qop = h2_ham

blocks = [MappedONVStateBlock(onv, JordanWigner()),
        TrotterAnsatzBlock(
            len(onv),
            qubit_uccsd,
            symbols,
            steps=1,
            time=0.1,
            order=1,
            grouping=FullyCommuting(),
            imaginary=True
        )
    ]
ket = CompositeBlock(blocks).build()
ket.plot(decompose_boxes=True, scrollable=True)

# VQE
initial_parameters = np.random.random(len(ket.symbols))
options = {"maxiter": 15}
optimizer = ScipyOptimizer(method="COBYLA", options=options)
vqe = VQE(
        operator=qop,
        ket=ket,
        primitive=StateVector(),
        engine=QarpEngine(),
        initial_parameters=initial_parameters,
        optimizer=optimizer,
        verbose=False,
).build()

e_vqe, x_vqe = vqe.run()
print("VQE Energy: ", e_vqe)

ham_mat = qop.sparse_matrix().toarray()

eigenvalues, eigenvectors = eigh(ham_mat)
h2_gs = eigenvectors[:, np.argmin(eigenvalues)]
h2_gse = eigenvalues[np.argmin(eigenvalues)]

# Overlap of VQE with exact GS (sparse_matrix and the qarpx statevector share the LSB basis)
params_vqe = vqe.optimal_parameters  # result.x mapped onto the sorted symbol tuple (§17)
h2_wfn_vqe = ket.set_symbols(params_vqe)
h2_wfn_vqe_SV = np.array(qx.QarpSimulator().statevector(h2_wfn_vqe.flatten(), h2_wfn_vqe.n_qubits))

print("overlap of VQE with exact GS: ", h2_wfn_vqe_SV @ h2_gs)

#### $N_2$ molecule

<div style="background-color: #fcc651e8; border-left: 4px solid #ff9900ff; padding: 6px; margin: 10px 0; color: black;">
<strong>⚠️ Note:</strong> With this example, we use MonteCarlo to follow up to a non-converged VQE result
</div>

In [ ]:
bl = 1.
geometry = f"N 0 0 0; N 0 0  {bl}"
mol = gto.M(atom=geometry, basis="sto3g", verbose=-1, symmetry=True)
mol.build()
mf = scf.RHF(mol)
mf.kernel()
mol.build()

converged_SCF_energy = -7.86217481976376
integrals, onv = active_space_from_mf(mf, 2, 3)

fermion_operator = restricted_integrals_to_fermion_operator(*integrals)

qop = JordanWigner().encode_operator(fermion_operator)

fucc, symbols = ucc_singles_and_doubles(onv, spin_conserving=True, generalised=True)
qucc = JordanWigner().encode_operator(fucc)

blocks = [MappedONVStateBlock(onv, JordanWigner()),
        TrotterAnsatzBlock(
            len(onv),
            qucc,
            symbols,
            steps=1,
            time=1,
            order=1,
            grouping=FullyCommuting(),
            imaginary=True
        )
    ]
ket = CompositeBlock(blocks).build()
ket.plot(decompose_boxes=True, scrollable=True)

# VQE
initial_parameters = np.random.random(len(ket.symbols))
options = {"maxiter": 80}
optimizer = ScipyOptimizer(method="COBYLA", options=options)
vqe = VQE(
        operator=qop,
        ket=ket,
        primitive=StateVector(),
        engine=QarpEngine(),
        initial_parameters=initial_parameters,
        optimizer=optimizer,
        verbose=False,
).build()

e_vqe, x_vqe = vqe.run()
print("VQE Energy: ", e_vqe)

ham_mat = qop.sparse_matrix().toarray()

eigenvalues, eigenvectors = eigh(ham_mat)
h2_gs = eigenvectors[:, np.argmin(eigenvalues)]
h2_gse = eigenvalues[np.argmin(eigenvalues)]

# Overlap of VQE with exact GS (sparse_matrix and the qarpx statevector share the LSB basis)
params_vqe = vqe.optimal_parameters  # result.x mapped onto the sorted symbol tuple (§17)
h2_wfn_vqe = ket.set_symbols(params_vqe)
h2_wfn_vqe_SV = np.array(qx.QarpSimulator().statevector(h2_wfn_vqe.flatten(), h2_wfn_vqe.n_qubits))

print("overlap of VQE with exact GS: ", h2_wfn_vqe_SV @ h2_gs)

## Monte Carlo simulation

We have computed the overlap $<\Psi_0|\phi_{vqe}>$ between the real ground state and the ground state found through VQE to check how much of a correction we will be introducing with Monte Carlo

We now start the proper Quantum Monte Carlo algorithm. First we apply the U transformation on all the basis state in the computational basis. This will give us the set of walker states. 

For the sake of clarity, and for aiding their manipulation, the walkers state are collected as tuples of this form:

$ walker_{i}=(|\phi_i> , sign(|\phi_i>), "i")$

where the elements of the tuple are, respectively, the walker wavefunction, its sign (initialized to be $1$) and its label. 

N.B. In principle the QMC does not different trajectory, as the convergence towards the target state is guaranteed by the spawning/death-cloning process. So the usual way of running it requires one sample and N walkers (with N large depending on the overlap and the complexity of the system). However non-sparse Hamiltonian are harder compuationally so it might be computationally more efficient to consider non zero classical trajectories (samples greater than zero) and a limited number of walkers.

In [ ]:
from qarp.algorithms import generate_states_new_basis
 
U = vqe.final_block.blocks[1].set_symbols(params_vqe)
# The walker basis must contain the reference state, so use the ONV's Hamming
# weight (2 for the H2/N2 examples, 3 for the Fermi-Hubbard one).
hamming_weight = int(np.sum(onv))
walker_states, walkers_circ, idxs = generate_states_new_basis(U, hamming_weight = hamming_weight)

# You can specify more than one Hamming weight to be included in your set of states by typing, eg for 2 and 3 
# walker_states, walkers_circ, idxs = generate_states_new_basis(U, hamming_weight = [2, 3] )

# Use NamedTuple for WalkerState
walker_states_lab = [
    WalkerState(state_data=j, sign=1, label=str(idxs[i])) 
    for i, j in enumerate(walker_states)
]

walker_circ_lab = [
    WalkerState(state_data=j, sign=1, label=str(idxs[i])) 
    for i, j in enumerate(walkers_circ)
]


# Show the walker basis: position, integer index, occupation-number bitstring (ONV)
# and its Hamming weight (number of occupied orbitals).
n_orb = len(onv)  # number of spin-orbitals = bitstring width

print(f"{'#':>3}  {'index':>5}  {'ONV':<{n_orb}}  {'weight':>6}")
print("-" * (3 + 2 + 5 + 2 + n_orb + 2 + 6))

for i, ws in enumerate(walker_states_lab):
    idx = int(ws.label)
    onv_str = "".join(str(b) for b in label_to_bits(idx, n_orb))  # position q = qubit q occupation
    print(f"{i:>3}  {idx:>5}  {onv_str:<{n_orb}}  {bin(idx).count('1'):>6}")

We evaluate the overlap between the walker set and the ground state from vqe, to identfy the corresponding label, i.e. the largest overlap corresponds to that state.

In [ ]:
# qarpx is LSB: qubit q carries onv[q] (bits_to_label packs LSB-first),
# matching the LSB walker labels returned by generate_states_new_basis.
ground_vqe_index = bits_to_label(onv) # reference state from VQE in the new basis has the same index as the original onv

In [ ]:
print("Ground VQE index:", ground_vqe_index)
print("Energy from VQE:", e_vqe)

In [ ]:
# Demo-sized parameters - increase T, samples, and the walker counts for tighter statistics.
T = 4
delta_tau = 0.1
N0 = 300
csi = 0.1
threshold = 200
approx_gs_energy = e_vqe
samples = 2

# Initialize the simulator
simulator = MonteCarlo(
    hamiltonian = qop,
    approx_ground_state_energy = approx_gs_energy, # generally the vqe energy
    total_time = T,
    time_step = delta_tau,
    initial_walker_count = N0,
    shift_damping = csi,
    reference_walker_label = ground_vqe_index,
    unitary_block = U.build(),
    walker_basis = walker_circ_lab,
    population_threshold = threshold,
    num_trajectories = samples,
    mode = "Quantum", # "Quantum" or "Semiclassical"
    primitive = TermwiseHadamardTest(), # e.g., "Statevector" or "TermwiseHadamardTest"
    n_shots = 2000,
    save_walker_history = True,
    history_save_interval = -1, # -1 to saves last step only
    verbose=False,
)
    
# BUILD - Build the simulator with U and walker states
simulator.build()
    
# Run the simulation
final_energy = simulator.run()

trajectories_smc = simulator.energy_estimates_trajectories
    
print("GS_energy:", np.real(final_energy))

In [ ]:
## Using qdrift to approximate the Hamiltonian

# Parameters
# Demo-sized parameters - increase T, samples, and the walker counts for tighter statistics.
T = 4
delta_tau = 0.1
N0 = 300
csi = 0.1
threshold = 200
approx_gs_energy = e_vqe
samples = 3

# Initialize the simulator (without U)
simulator = MonteCarlo(
    hamiltonian = qop,
    approx_ground_state_energy = approx_gs_energy,
    total_time = T,
    time_step = delta_tau,
    initial_walker_count = N0,
    reference_walker_label = ground_vqe_index,
    unitary_block = U.build(),
    walker_basis = walker_states_lab,
    shift_damping = csi,
    population_threshold = threshold,
    mode = "Semiclassical", # "quantum" or "semiclassical"
    primitive = StateVector(), # e.g., "Statevector" or "TermwiseHadamardTest"
    save_walker_history = True,
    history_save_interval = -1, # -1 to saves last step only
    qdrift = True, #Qdrift the Hamiltonian i.e. sampling the most probable term only
    qdrift_samples = 100, #number of sampling with Qdrift
    qdrift_ratio = 0.6, #ratio of the sampling, if 0.5, half of the terms are deterministically added, the other half sampled
    verbose = False,
)
    
# BUILD - Build the simulator with U and walker states
simulator.build()
    
# Run the simulation
final_energy = simulator.run()

trajectories = simulator.energy_estimates_trajectories
    
print("GS_energy:", np.real(final_energy))


## Plots

In [ ]:
import matplotlib.pyplot as plt

trajectories_smc_plot = [trajectories[0] for trajectories in trajectories_smc]
trajectories_smc_plot=np.array(trajectories_smc_plot)
tr_mean_smc_plot=np.mean(trajectories_smc_plot,axis=0)

t=np.linspace(0,T,int(T/delta_tau))
plt.plot(t,trajectories_smc_plot.T,linestyle=":")
plt.plot(t,tr_mean_smc_plot,color="black", linewidth=2,label="Estimated GS (mean)-semiclassical")
plt.hlines(e_vqe,xmin=0,xmax=T,color="red",label="Initial VQE GS energy")
plt.hlines(h2_gse,xmin=0,xmax=T,color="pink",label="Real GS")
plt.legend(loc=0)


plt.ylabel("GS Energy")
plt.xlabel("T")

In [ ]:
import matplotlib.pyplot as plt

trajectories_plot = [traj[0] for traj in trajectories]
trajectories_plot=np.array(trajectories_plot)
tr_mean_plot=np.mean(trajectories_plot,axis=0)

t=np.linspace(0,T,int(T/delta_tau))
plt.plot(t,trajectories_plot.T,linestyle=":")#
plt.plot(t,tr_mean_plot,color="black", linewidth=2,label="Estimated GS (mean)-quantum_drift")
plt.hlines(e_vqe,xmin=0,xmax=T,color="red",label="Initial VQE GS energy")
plt.hlines(h2_gse,xmin=0,xmax=T,color="pink",label="Real GS")
plt.legend(loc=0)


plt.ylabel("GS Energy")
plt.xlabel("T")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from typing import List, Tuple
 
def plot_walker_histogram(t_walk: List[Tuple[np.ndarray, float, str]], title: str = "Walker Occupancy Histogram"):
        """
        Generates distribution plot of walker occupancy.

        Args:
            walk: A list of tuples (array, sign, label)
        """

        # Extract labels and convert to integers
        occ_walk = [int(lab)for _, _, lab in t_walk]
        #print(occ_walk)

        # Create the histogram
        plt.figure(figsize=(10, 6))  # Adjust figure size for better spacing
        n, bins, patches = plt.hist(occ_walk, bins=np.arange(min(occ_walk)-0.5, max(occ_walk) + 1.5, 1), edgecolor='black', linewidth=1.2,  rwidth=0.8) # Use integer bins

        # Customize the plot
        plt.title(title, fontsize=16)
        plt.xlabel("State Label", fontsize=14)
        plt.ylabel("Occupancy", fontsize=14)

        plt.xticks(fontsize=12)
        plt.yticks(fontsize=12)
        plt.tick_params(direction='out', length=6, width=2)  # Improve tick appearance

        for patch in patches:
            patch.set_alpha(0.8)  # Make bars slightly transparent
            patch.set_facecolor(plt.cm.viridis(patch.get_height() / max(n))) #Color based on height

        plt.tight_layout()

        plt.show()

plot_walker_histogram(simulator.walker_history[0][-1])


Monte Carlo can also find solutions Hamiltonians from graph optimization problems. Specifically, the pipeline is very similar for Max Cut problems with a cardinality constraint, which is implemented by means of the Hamming weight in the used bistrings. Let's see that in the following snippets

In [ ]:
import networkx as nx
import qarp
from qarp.operators import QubitOperator
qarp.config.seed = 1234


n_nodes = 5         # change this for number of nodes == n_qubits
p_edges = .8        # change this for probability of edges. Higher -> more edges. 

graph = nx.erdos_renyi_graph(n=n_nodes, p=p_edges, seed=qarp.config.seed)

## Max Cut formulation - QMC Hamiltonian
ham = QubitOperator()
for q0, q1, data in graph.edges(data=True):
    coef = data["weight"] if len(data) > 0 else 1
    ham += QubitOperator("Z" + str(q0) + " Z" + str(q1), coef)

## We define a cardinality constraint by adding linear terms
n_ones = 2  # modify this to change the cardinality constraint == hamming weight 


We can perform state preparation in a similar fashion as in the chemistry case, knowing that UCC excitations conserve the Hamming weight

In [ ]:
# Starting bitstring
bitstring = [1]*n_ones + [0]*(n_nodes - n_ones) 



# We are going to use the UCC excitation functionality to build the needed objects to run VQE
# It works when using Jordan-Wigner, as the Hamming weight is conserved for this mapping
onv = bitstring
fermionic_uccsd, symbols = ucc_singles_and_doubles(
        onv, spin_conserving=False, generalised=False) # Now we do not conserve spin, as in the chemistry case
qubit_uccsd = JordanWigner().encode_operator(fermionic_uccsd)

blocks = [MappedONVStateBlock(onv, JordanWigner()),
        TrotterAnsatzBlock(
            len(bitstring),
            qubit_uccsd,
            symbols,
            steps=1,
            time=0.1,
            order=1,
            grouping=FullyCommuting(),
            imaginary=True
        )
    ]
ket = CompositeBlock(blocks).build()

# VQE
print("Running VQE...")
initial_parameters = np.random.random(len(ket.symbols))
options = {"maxiter": 15}
optimizer = ScipyOptimizer(method="COBYLA", options=options)
vqe = VQE(
        operator=ham,
        ket=ket,
        primitive=StateVector(),
        engine=QarpEngine(),
        initial_parameters=initial_parameters,
        optimizer=optimizer,
        verbose=False,
).build()

e_vqe, x_vqe = vqe.run()
print("VQE Energy: ", e_vqe)

ham_mat = ham.sparse_matrix().toarray()

eigenvalues, eigenvectors = eigh(ham_mat)
gs = eigenvectors[:, np.argmin(eigenvalues)]
gse = eigenvalues[np.argmin(eigenvalues)]

# Overlap of VQE with exact GS (sparse_matrix and the qarpx statevector share the LSB basis)
params_vqe = vqe.optimal_parameters  # result.x mapped onto the sorted symbol tuple (§17)
wfn_vqe = ket.set_symbols(params_vqe)
wfn_vqe_SV = np.array(qx.QarpSimulator().statevector(wfn_vqe.flatten(), wfn_vqe.n_qubits))

print("overlap of VQE with exact GS: ", wfn_vqe_SV @ gs)

We extract the unitary and run Monte Carlo

In [ ]:
U = vqe.final_block.blocks[1].set_symbols(params_vqe)
walker_states, walkers_circ, idxs = generate_states_new_basis(U, hamming_weight = n_ones)

walker_circ_lab = [
    WalkerState(state_data=j, sign=1, label=str(idxs[i])) 
    for i, j in enumerate(walkers_circ)
]

In [ ]:
# Running Monte Carlo now
print("Running Monte Carlo...")
# qarpx is LSB: qubit q carries onv[q] (bits_to_label packs LSB-first),
# matching the LSB walker labels returned by generate_states_new_basis.
ground_vqe_index = bits_to_label(onv) # reference state from VQE in the new basis has the same index as the original onv
# Demo-sized parameters - increase T, samples, and the walker counts for tighter statistics.
T = 6
delta_tau = 0.1
N0 = 300
csi = 0.1
threshold = 200
approx_gs_energy = e_vqe
samples = 3


# Initialize the simulator
simulator = MonteCarlo(
    hamiltonian = ham,
    approx_ground_state_energy = approx_gs_energy, # generally the vqe energy
    total_time = T,
    time_step = delta_tau,
    initial_walker_count = N0,
    shift_damping = csi,
    reference_walker_label = ground_vqe_index,
    unitary_block = U.build(),
    walker_basis = walker_circ_lab,
    population_threshold = threshold,
    num_trajectories = samples,
    mode = "Quantum", # "Quantum" or "Semiclassical"
    primitive = TermwiseHadamardTest(), # e.g., "Statevector" or "TermwiseHadamardTest"
    n_shots = None,
    save_walker_history = True,
    history_save_interval = -1, # -1 to saves last step only
    verbose=True,
)
    
# BUILD - Build the simulator with U and walker states
simulator.build()
    
# Run the simulation
final_energy = simulator.run()

trajectories_smc = simulator.energy_estimates_trajectories

print("GS_energy:", np.real(final_energy))

In [ ]:
trajectories_smc_plot = [trajectories[0] for trajectories in trajectories_smc]
trajectories_smc_plot=np.array(trajectories_smc_plot)
tr_mean_smc_plot=np.mean(trajectories_smc_plot,axis=0)

t=np.linspace(0,T,int(T/delta_tau))
plt.plot(t,trajectories_smc_plot .T,linestyle=":")#
plt.plot(t,tr_mean_smc_plot,color="black", linewidth=2,label="Estimated GS (mean)-quantum_drift")
plt.hlines(e_vqe,xmin=0,xmax=T,color="red",label="Initial VQE GS energy")
plt.hlines(gse,xmin=0,xmax=T,color="pink",label="Real GS")
plt.legend(loc=0)


plt.ylabel("GS Energy")
plt.xlabel("T")

plot_walker_histogram(simulator.walker_history[0][-1])


In [ ]:
from collections import Counter


counter_solutions =  Counter( [int(i.label) for i in simulator.walker_history[0][-1]] )
num_solutions = 6

print("Ground VQE index (we should not consider this state as the optimal solution as it was used as an ansatz): ", ground_vqe_index)

print("Most likely solutions and corresponding energy")
# idx is an LSB walker label (qubit q = bit q) — the same basis as ham_mat.
for idx, num_walkers in counter_solutions.most_common(num_solutions):
    bitstr = "".join(str(b) for b in label_to_bits(idx, n_nodes))  # qubit q = bit q (qarpx LSB)
    print(idx, bitstr, ham_mat[idx, idx])

We now solve for multiple excited states

In [ ]:
from qarp.algorithms import generate_states_new_basis
 
U = vqe.final_block.blocks[1].set_symbols(params_vqe)
walker_states, walkers_circ, idxs = generate_states_new_basis(U)

# You can specify more than one Hamming weight to be included in your set of states by typing, eg for 2 and 3 
# walker_states, walkers_circ, idxs = generate_states_new_basis(U, hamming_weight = [2, 3] )

# Use NamedTuple for WalkerState
walker_states_lab = [
    WalkerState(state_data=j, sign=1, label=str(idxs[i])) 
    for i, j in enumerate(walker_states)
]

walker_circ_lab = [
    WalkerState(state_data=j, sign=1, label=str(idxs[i])) 
    for i, j in enumerate(walkers_circ)
]


# Show the walker basis: position, integer index, occupation-number bitstring (ONV)
# and its Hamming weight (number of occupied orbitals).
n_orb = len(onv)  # number of spin-orbitals = bitstring width

print(f"{'#':>3}  {'index':>5}  {'ONV':<{n_orb}}  {'weight':>6}")
print("-" * (3 + 2 + 5 + 2 + n_orb + 2 + 6))

for i, ws in enumerate(walker_states_lab):
    idx = int(ws.label)
    onv_str = "".join(str(b) for b in label_to_bits(idx, n_orb))  # position q = qubit q occupation
    print(f"{i:>3}  {idx:>5}  {onv_str:<{n_orb}}  {bin(idx).count('1'):>6}")

In [ ]:
num_target_states = 3
n_qubits = U.n_qubits

# ws[0] is a qarpx LSB statevector — the same basis as ham_mat.
engy = []
for ws in walker_states_lab:
    psi = ws[0]
    engy.append(np.real(psi.conj().T @ ham_mat @ psi))
energies_vqe = sorted(engy)

reference_state_label = []
for j in range(num_target_states):
    reference_state_label.append(int(walker_states_lab[engy.index(energies_vqe[j])].label))

print("VQE Reference Labels:", reference_state_label)
print("Energies from VQE:", energies_vqe[0:num_target_states])
print("Target energies:", eigenvalues[0:num_target_states] )

In [ ]:
# Demo-sized parameters - increase T, samples, and the walker counts for tighter statistics.
T = 6
delta_tau = 0.1
N0 = 200
csi = [0.1]*num_target_states
threshold = 500
approx_gs_energy = energies_vqe[0:num_target_states]
samples = 3

# Initialize the simulator
simulator = MonteCarlo(
    hamiltonian = ham,
    num_target_states=num_target_states,
    approx_ground_state_energy = approx_gs_energy, # generally the vqe energy
    total_time = T,
    time_step = delta_tau,
    initial_walker_count = [N0]*(num_target_states-1)+[1000],
    shift_damping = csi,
    reference_walker_label = reference_state_label,
    unitary_block = U.build(),
    walker_basis = walker_states_lab,
    population_threshold = threshold,
    num_trajectories = samples,
    mode = "Semiclassical", # "Quantum" or "Semiclassical"
    primitive = StateVector(), # e.g., "StateVector" or "TermwiseHadamardTest"
    n_shots = 5000,
    save_walker_history = True,
    history_save_interval = -1, # -1 to saves last step only
    verbose=True,
)
    
# BUILD - Build the simulator with U and walker states
simulator.build()
    
# Run the simulation
final_energy = simulator.run()

trajectories_smc = simulator.energy_estimates_trajectories
    
print("GS_energy:", np.real(final_energy))

In [ ]:
import matplotlib.pyplot as plt

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

fig, axes = plt.subplots(num_target_states, 1, figsize=(10, 4 * num_target_states), sharex=True)
if num_target_states == 1:
    axes = [axes]

t = np.linspace(0, T, int(T / delta_tau))

for j, ax in enumerate(axes):
    c = colors[j % len(colors)]
    trajectories_j = np.array([trajectory[j] for trajectory in trajectories_smc])
    tr_mean = np.mean(trajectories_j, axis=0)

    ax.plot(t, trajectories_j.T, linestyle=":", color=c, alpha=0.4)
    ax.plot(t, tr_mean, color=c, linewidth=2.5, label=f"Mean estimate — state {j}")
    ax.axhline(energies_vqe[j], color="red", linestyle="--", linewidth=1.5, label=f"VQE energy (state {j}): {energies_vqe[j]:.4f}")
    ax.axhline(eigenvalues[j], color="green", linestyle="-.", linewidth=1.5, label=f"Exact energy (state {j}): {eigenvalues[j]:.4f}")
    ax.set_ylabel("Energy")
    ax.set_title(f"State {j}", fontsize=11)
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Imaginary time $\\tau$")
fig.suptitle("Monte Carlo energy trajectories", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
